# Prefill: TTFT vs prompt length

## Experiment design

Source: `data/raw/news_all.parquet`, 1,407,328 Benzinga headlines, one ticker tag per row, 6,204 tickers (from hugging face, [`scripts/download_headlines_corpus.py`](../scripts/download_headlines_corpus.py)).

**Signal**: headlines for the ticker being asked about.

| Ticker | Rows | Selection |
|---|---|---|
| WMT | 10 | All rows with stock column set to WMT.|
| AMZN | 7 | All rows with stock column set to AMZN. |

**Noise**: sets prompt length. Consists of N rows from `news_all` where the stock column is NOT the chosen ticker (`no_WMT_news.parquet`, `no_AMZN_news.parquet`). N ∈ {50, 100, 200, 300, 500} and 750 (Qwen3-4B) or 1000 (Gemma 4 E2B). Qwen3-4B stops at 750: AMZN at 1,000 returned no tokens (hardware insufficient).

**Prompt**: identical for both tickers apart from `{ticker}` and the headlines.

| # | Role | Content |
|---|---|---|
| 1 | system | `[request <8 hex>] You are a quantitative financial assistant. For the requested ticker, call get_stock_news tool, then generate your response.` |
| 2 | user | `Based only on the headlines, will the price target for {ticker} go up?` |
| 3 | assistant | tool call `get_stock_news({"ticker": "<ticker>"})`, pre-filled |
| 4 | tool | noise then signal headlines, one per line |
| 5 | user | `Using only the tool results, answer as JSON: a confidence between 0 and 1, and a summary that directly answers what was asked.` |

The request id is new per request, so no KV is reused across requests.

**Measurement**: streaming, `prefill_only: true`, `max_tokens: 2`, `enable_thinking: true`. TTFT = arrival of the chunk carrying `finish_reason`, i.e. when token `max_tokens` = 2 has been generated: prefill + 1 decode step. Not "first visible text": the reasoning parser hides the first token for Qwen3 (`<think>`) and the first ~7 for Gemma 4 (`<|channel>`, `thought`, …). One server per config; continuous batching off; prefix cache off; concurrency 1; 1 warm-up, then 5 repeats per noise level.


## Performance of each phase of an LLM, starting with prefill

- **Prefill (this benchmark):** n prompts multiplied with weight matrices; builds the KV cache. Weights ~ n·d², attention ~ n²·d. Compute-bound.
- **Decode (TODO):** one token per step, re-reading weights and the whole KV cache. Memory-bandwidth-bound.
- **Hardware considerations:** which GPU, which precision, memory requirements, how much context to allow.

### Investigating prefill: where the KV cache is built and attention grows as n²

First, naive version: all headlines and any question gave poor answers and no control over length.

## TTFT scales with prompt length non-linearly

Qwen3-4B (4-bit) on vllm-mlx · Apple M1, 8 GB · mean of 5 repeats per point, error bars 95% confidence interval · prefix cache off, unique prompt per request.

| Ticker | Signal headlines | Noise headlines | Data | Config |
|---|---|---|---|---|
| WMT | 10 | 50, 100, 200, 300, 500, 750 | [`results/vllm-mlx-reps/ttft-prefill-baseline-qwen3-wmt.csv`](results/vllm-mlx-reps/ttft-prefill-baseline-qwen3-wmt.csv) | [`configs/vllm-mlx-reps/ttft-prefill-baseline-qwen3-wmt.yaml`](configs/vllm-mlx-reps/ttft-prefill-baseline-qwen3-wmt.yaml) |
| AMZN | 7 | 50, 100, 200, 300, 500, 750 | [`results/vllm-mlx-reps/ttft-prefill-baseline-qwen3-amzn.csv`](results/vllm-mlx-reps/ttft-prefill-baseline-qwen3-amzn.csv) | [`configs/vllm-mlx-reps/ttft-prefill-baseline-qwen3-amzn.yaml`](configs/vllm-mlx-reps/ttft-prefill-baseline-qwen3-amzn.yaml) |

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

R = "results/vllm-mlx-reps/"
# t(0.975, dof): 95% interval half-width = T95[dof] * standard error
T95 = {1: 12.71, 2: 4.30, 3: 3.18, 4: 2.78, 5: 2.57, 6: 2.45, 7: 2.36, 8: 2.31, 9: 2.26}
RUNS = {"WMT": R + "ttft-prefill-baseline-qwen3-wmt.csv", "AMZN": R + "ttft-prefill-baseline-qwen3-amzn.csv"}


def points(path):
    """Per noise level: prompt tokens, mean TTFT and its 95% interval half-width over repeats."""
    df = pd.read_csv(path).dropna(subset=["ttft_s"])
    g = df[df.ok].groupby("noise_headlines")
    n = g.size()
    return pd.DataFrame({"prompt_tokens": g.prompt_tokens.median(), "ttft_s": g.ttft_s.mean(),
                         "ci95_s": n.map(lambda k: T95[k - 1]) * g.ttft_s.std() / np.sqrt(n), "reps": n})


def fit(m):
    """t = b·n + c·n² on the per-level means, least squares on relative error.
    b_ci, c_ci: 95% half-widths from the fit residuals (6 points, 4 dof)."""
    n, t = m.prompt_tokens.to_numpy(), m.ttft_s.to_numpy()
    A = np.c_[n, n**2] / t[:, None]
    (b, c) = coef = np.linalg.lstsq(A, np.ones_like(t), rcond=None)[0]
    r = 1 - A @ coef
    dof = len(t) - 2
    b_ci, c_ci = T95[dof] * np.sqrt(np.diag(r @ r / dof * np.linalg.inv(A.T @ A)))
    pred = b * n + c * n**2
    r2 = 1 - ((t - pred) ** 2).sum() / ((t - t.mean()) ** 2).sum()
    return {"b": b, "c": c, "b_ci": b_ci, "c_ci": c_ci, "r2": r2, "resid_%": pd.Series(100 * (t / pred - 1), m.index)}


def report(name, f, m):
    n, t = m.prompt_tokens, m.ttft_s
    print(f"{name:4}  t ≈ ({f['b'] * 1e3:.3f} ± {f['b_ci'] * 1e3:.3f}) ms·n + ({f['c'] * 1e6:.4f} ± {f['c_ci'] * 1e6:.4f}) µs·n²   "
          f"R² = {f['r2']:.5f}   n* = b/c ≈ {f['b'] / f['c']:,.0f}   {n.iloc[0] / t.iloc[0]:.0f} → {n.iloc[-1] / t.iloc[-1]:.0f} tokens/s")


pts = {k: points(p) for k, p in RUNS.items()}
fits = {k: fit(m) for k, m in pts.items()}
for k in fits:
    report(k, fits[k], pts[k])

b, c = fits["WMT"]["b"], fits["WMT"]["c"]
x = np.linspace(0, 1.05 * pts["WMT"].prompt_tokens.max(), 200)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, b * x + c * x**2, label="fit (WMT)")
ax.plot(x, b * x, "--", label="linear (if every token cost the same)")
ax.errorbar(pts["WMT"].prompt_tokens, pts["WMT"].ttft_s, yerr=pts["WMT"].ci95_s, fmt="ko", capsize=3, label="WMT")
ax.errorbar(pts["AMZN"].prompt_tokens, pts["AMZN"].ttft_s, yerr=pts["AMZN"].ci95_s, fmt="s", mfc="none", mec="C3", ecolor="C3", ms=9, capsize=3, label="AMZN")
ax.set(xlabel="Prompt length (tokens)", ylabel="Time to first token (s)", xlim=(0, None), ylim=(0, None))
ax.grid(alpha=0.3)
ax.legend();

### Fit quality

Per noise level: 95% interval of the mean and residual from the fit, both as % of the time.

In [ ]:
def quality(pts, fits):
    return pd.concat({k: pd.DataFrame({"prompt_tokens": m.prompt_tokens, "ci95_%": 100 * m.ci95_s / m.ttft_s,
                                       "resid_%": fits[k]["resid_%"]}) for k, m in pts.items()}, axis=1)


quality(pts, fits).round(2)

- R² = 0.99986 (WMT), 0.99984 (AMZN). TTFT spans 9–166 s, so any smooth curve gives R² near 1; the residuals are the informative check.
- Residuals −1.2% to +2.1%, same pattern for both tickers: high at both ends, low in the middle. The 95% intervals of the means are 0.02–0.3%: the misfit is the model, not repeat noise.
- A constant term fits it: t = a + b·n + c·n² gives a = 0.61 s and residuals within ±0.3% (WMT). Not investigated; candidates are per-request overhead and the one decode step in TTFT.
- Coefficient intervals come from the fit residuals, so they measure the misfit: b ±3%, c ±12% (WMT).

## Check: TTFT depends on token count, not content

AMZN vs WMT: different signal (7 vs 10 headlines, +47–49 tokens), same noise, separate runs on the same day.

- Fit coefficients agree: b within 0.6%, c within 1.6%, inside their 95% intervals.
- AMZN means are within −1.4% to +1.1% of the WMT fit (below).
- AMZN − WMT gap per noise level: measured 0.21–1.00 s, predicted from the extra tokens 0.33–0.62 s. The differences exceed the within-run intervals (≤ 0.12 s): the tickers ran separately, and within-run intervals do not cover run-to-run drift. Interleave tickers in one run to resolve it.
- Noise is shared. Signal is 155 tokens (WMT) and 203 (AMZN): 11–14% of the prompt at N = 50, ~1% at the top. Qwen3-4B is dense: FLOPs depend on token count only.

In [ ]:
b, c = fits["WMT"]["b"], fits["WMT"]["c"]
amzn = pts["AMZN"][["prompt_tokens", "ttft_s", "ci95_s"]].assign(wmt_fit_s=lambda m: b * m.prompt_tokens + c * m.prompt_tokens**2)
amzn["diff_%"] = 100 * (amzn.ttft_s / amzn.wmt_fit_s - 1)
amzn.round(2)

## Gemma 4 E2B

Same prompts, data, settings and TTFT rule as Qwen3-4B. Model: `mlx-community/Gemma4-E2B-IT-Text-int4`.

| Ticker | Signal headlines | Noise headlines | Data | Config |
|---|---|---|---|---|
| WMT | 10 | 50, 100, 200, 300, 500, 1000 | [`results/vllm-mlx-reps/ttft-prefill-baseline-gemma4-e2b-wmt.csv`](results/vllm-mlx-reps/ttft-prefill-baseline-gemma4-e2b-wmt.csv) | [`configs/vllm-mlx-reps/ttft-prefill-baseline-gemma4-e2b-wmt.yaml`](configs/vllm-mlx-reps/ttft-prefill-baseline-gemma4-e2b-wmt.yaml) |
| AMZN | 7 | 50, 100, 200, 300, 500, 1000 | [`results/vllm-mlx-reps/ttft-prefill-baseline-gemma4-e2b-amzn.csv`](results/vllm-mlx-reps/ttft-prefill-baseline-gemma4-e2b-amzn.csv) | [`configs/vllm-mlx-reps/ttft-prefill-baseline-gemma4-e2b-amzn.yaml`](configs/vllm-mlx-reps/ttft-prefill-baseline-gemma4-e2b-amzn.yaml) |

AMZN completes at 1,000 noise headlines on Gemma (failed on Qwen3-4B). Prompt token counts differ slightly from Qwen's: different tokenizer.

### Architecture (from `config.json`)

| | Qwen3-4B | Gemma 4 E2B |
|---|---|---|
| Parameters | ~4.0B | ~4.6B total, ~2B "effective"; the rest are per-layer embedding tables, looked up, not multiplied |
| Layers / hidden size | 36 / 2560 | 35 / 1536 |
| Attention | full, all 36 layers | sliding window (512 tokens) in 28 layers; full in 7 (layers 4, 9, …, 34) |
| Query / KV heads | 32 / 8, head dim 128 | 8 / 1, head dim 256 (512 in full layers) |
| KV sharing | none | layers 15–34 reuse K/V from earlier layers; no K/V projection |
| MLP | 9728 wide | 6144 wide; 12288 in layers 15–34 |
| Weights | 4-bit, group 64, fp16 scales/biases | same |

**Prefill skips most of the model.** Layers 15–34 write no KV cache, so prompt positions are not needed past layer 14. mlx-vlm (which vllm-mlx uses to load Gemma 4) prefills with `logits_to_keep=1` and trims the hidden state to the last token at layer 15. Only layers 0–14 run over the whole prompt.

In [ ]:
G_RUNS = {"WMT": R + "ttft-prefill-baseline-gemma4-e2b-wmt.csv", "AMZN": R + "ttft-prefill-baseline-gemma4-e2b-amzn.csv"}
g_pts = {k: points(p) for k, p in G_RUNS.items()}
g_fits = {k: fit(m) for k, m in g_pts.items()}
for k in g_fits:
    report(k, g_fits[k], g_pts[k])

gb, gc = g_fits["WMT"]["b"], g_fits["WMT"]["c"]
x = np.linspace(0, 1.05 * g_pts["WMT"].prompt_tokens.max(), 200)
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(x, gb * x + gc * x**2, label="fit (WMT)")
ax.plot(x, gb * x, "--", label="linear (if every token cost the same)")
ax.errorbar(g_pts["WMT"].prompt_tokens, g_pts["WMT"].ttft_s, yerr=g_pts["WMT"].ci95_s, fmt="ko", capsize=3, label="WMT")
ax.errorbar(g_pts["AMZN"].prompt_tokens, g_pts["AMZN"].ttft_s, yerr=g_pts["AMZN"].ci95_s, fmt="s", mfc="none", mec="C3", ecolor="C3", ms=9, capsize=3, label="AMZN")
ax.set(title="Gemma 4 E2B", xlabel="Prompt length (tokens)", ylabel="Time to first token (s)", xlim=(0, None), ylim=(0, None))
ax.grid(alpha=0.3)
ax.legend()
quality(g_pts, g_fits).round(2)

### Gemma4 performance

Fit form is the same: `t = b·n + c·n²`. Predictions from the table above, FLOPs counted for weights that run over every prompt token:

| | Qwen3-4B | Gemma 4 E2B | Ratio predicted | Ratio measured |
|---|---|---|---|---|
| Linear term `b`: weights over the full prompt | 36 layers, ~3.6B | layers 0–14, ~0.55B | ~6.6× | ~7.2× |
| Quadratic term `c`: full-attention layers over the full prompt | 36 × 4096 wide | 3 (layers 4, 9, 14) × 4096 wide | ~12× | ~23× |

**Gemma4 performance features**
- **Linear term.** ~7× cheaper per token, matched by the trimmed-prefill weight count. Achieved throughput is similar for both: ~2 FLOPs × weights × tokens/s ≈ 1.1 TFLOP/s (Qwen) and ≈ 1.25 TFLOP/s (Gemma), under the M1 GPU's ~2.6 TFLOP/s peak.
- **Quadratic term.** Sliding-window layers attend over at most 512 tokens: linear in n. Only 3 full-attention layers see the whole prompt. Measured `c` is ~2× lower than predicted; could it be tiled/sliding window attention and cache efficiency?. `c` is the weaker estimate: n² is ~18% of Gemma's TTFT at 20k tokens, against ~36% for Qwen at 17k.
- **Crossover** n* = b/c ≈ 96k tokens (Qwen ≈ 30k). Inside Gemma's 131,072 context but 5× past this sweep: over 1–20k tokens Gemma is close to linear.
- **Memory.** 1 KV head, KV for 15 layers only, and 512-token windows in most of those: a much smaller KV cache per token. AMZN at ~20k tokens completes; on Qwen it did not.
- **Fit quality.** R² = 0.99998 (WMT), 0.99997 (AMZN). Residuals −0.9% to +1.5%, 95% intervals of the means 0.04–0.4%. Coefficient intervals: b ±1%, c ±9% (WMT). A constant term is 0.02 s here, against 0.61 s for Qwen3-4B.
- **AMZN vs WMT.** b within 0.1%, c within 0.4%.

In [ ]:
# b_ms = coefficient b in the above fit in ms
# c_us = coefficient c in the above fit in us
# n_start = b_ms/c_us
# top_n = maximum input tokens benchmarked
# ttft_top_s = time to first token for maximum input tokens benchmarked
# quad_share_top_% = percentage of the quadratic term (with coefficient) over the predicted time for the maximum input tokens benchmark run

rows = []
for name, fs, ms in [("Qwen3-4B", fits, pts), ("Gemma 4 E2B", g_fits, g_pts)]:
    b_, c_ = fs["WMT"]["b"], fs["WMT"]["c"]
    n = ms["WMT"].prompt_tokens.iloc[-1]
    rows.append({"model": name, "b_ms": b_ * 1e3, "c_us": c_ * 1e6, "n_star": b_ / c_,
                 "top_n": n, "ttft_top_s": ms["WMT"].ttft_s.iloc[-1], "quad_share_top_%": 100 * c_ * n**2 / (b_ * n + c_ * n**2)})
cmp = pd.DataFrame(rows).set_index("model")
# cmp.loc["ratio Qwen/Gemma", ["b_ms", "c_us"]] = cmp.iloc[0][["b_ms", "c_us"]] / cmp.iloc[1][["b_ms", "c_us"]]
cmp.round(3)

## Prefill (compute bound) vs decode (memory bound)

| | Prefill: reading the prompt | Decode: writing the answer |
|---|---|---|
| Input | n prompt tokens, all at once | 1 new token |
| Attention (each of 36 layers) | every token vs every earlier one: n² | new token vs all n cached: n |
| Weight multiplies | same cost per token: n | all weights re-read for one token |
| KV cache (keys and values of every token so far) | writes one entry per prompt token | read in full each step, one entry added |
| Output | first token · time to first token | next token · repeat until done |

Simplified decoder-only transformer, reference Vaswani et al., “Attention Is All You Need”, NeurIPS 2017 ([arXiv:1706.03762](https://arxiv.org/abs/1706.03762)). Layer count is Qwen3-4B’s.

## Prefill and decode stress different hardware

| | Prefill: reading the prompt | Decode: writing the answer |
|---|---|---|
| Tokens per step | all n prompt tokens at once | one |
| Weight multiplies | matrix × matrix: each weight loaded once, used by n tokens | matrix × vector: every weight loaded again for each token |
| Attention | every token against every earlier one: grows as n² | the new token against n cached keys: grows as n |
| KV cache | written, one entry per prompt token: grows as n | read in full every step, one entry appended |
| Bottleneck | compute (FLOP/s) | memory bandwidth (bytes/s) |
| What users see | time to first token | milliseconds per output token |
| GPU feature that helps | tensor-core throughput, FP8/FP4 | HBM bandwidth (H200 about 4.8 TB/s) |

Measured so far: prefill. Decode sweep next, same harness.